In [1]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)
import librosa
import os
import matplotlib.pyplot as plt
import numpy as np
import pickle
import math
import time
import wave
import struct
import random
import gc
import scipy
import json
import re
from itertools import groupby
fea_dir='/workspace/mnt/slave3/tone_training_fea'# fn_fea63.npy

2026-03-28 16:30:06.518454: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-28 16:30:06.551131: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9360] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-28 16:30:06.551160: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-28 16:30:06.551184: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1537] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-28 16:30:06.557886: I tensorflow/core/platform/cpu_feature_g

In [2]:
aveoe4_target={
'M1':[5.469093047410694, 0.011012099966053098, -0.014568323858748868, -0.005783063882019955 ],#M1
'M2':[5.246173001693238, 0.004049068842379105, 0.024549176995476464, -0.009276374116975216 ],#M2
'M3':[5.1044128461888585, -0.1368410788426736, 0.023567859212565972, 0.02309820985224488 ],#M3
'M4':[5.359357407609216, -0.08578662996576089, -0.02803258719092904, 0.019774980150000898 ],#M4
'M5':[5.108925694677095, -0.07198157742009792, 0.0017029727276897432, 0.01638156068781905],#M5
'H31':[5.281164659497466, -0.128633454742137, 0.002124414947062, 0.019041328130274],#H31
'H5':[5.519714730277190, 0.011872408010772, -0.004542050079823, -0.003054793059757],#H5
'H24':[5.152172863235218, -0.008486718193493, 0.049115685492959, -0.011936673321675],#H24
'H11':[5.104523048656755, -0.130883285302238, 0.025056400507341, 0.017634886302532],#H11
'H2':[5.307759510367972, -0.096714767117002, 0.003526958815474, 0.012184974479543],#H2
'H55':[5.493415925128653, 0.018078665693827, -0.007093534929418, -0.005989730485095]#H55
}

In [3]:
with open('/workspace/home/cewarman/research/kaldi/CEmix/TAAN/data/train/wav.scp', 'r', encoding='utf8') as f:
    wavlst=["/workspace"+line.strip().split()[1] for line in f.readlines()]
with open('/workspace/mnt/slave3/HAT-corpus/Hakka_ASR_four_counties/wav.lst', 'r', encoding='utf8') as f:
    Hwavlst=["/workspace"+line.strip() for line in f.readlines()]
used_wavlst=[]
for i in range(len(wavlst)):
    if(wavlst[i].split('/')[-1][:3]=='SSB'):
        used_wavlst.append(wavlst[i])
for i in range(len(Hwavlst)):
    used_wavlst.append(Hwavlst[i])
wavlst=used_wavlst
print(len(wavlst))

193887


In [4]:
def fea_extractor(file_path):
    y, sr = librosa.load(file_path, mono=True, sr=16000)
    S_ir = librosa.stft(y=y, n_fft=512, hop_length=160)
    #print(np.abs(S_ir).max())
    ##print(np.abs(S_ir).shape,np.abs(S_ir))
    S_dB = librosa.amplitude_to_db(np.abs(S_ir), ref=np.max)
    #fig, ax = plt.subplots()
    #img = librosa.display.specshow(S_dB, x_axis='time',
    #                     y_axis='mel', sr=sr,
    #                     fmax=16000, ax=ax)
    #fig.colorbar(img, ax=ax, format='%+2.0f dB')
    #ax.set(title='Mel-frequency spectrogram')
    return S_dB.transpose()


In [5]:
os.system("mkdir -p {}".format(fea_dir))
for i in range(len(wavlst)):
    fn=os.path.basename(wavlst[i]).split('.')[0]
    
    out_fn="{}/{}_fea63".format(fea_dir,fn)
    if(os.path.isfile(out_fn+'.npy')):
        continue
    else:
        fea=fea_extractor(wavlst[i])[:,1:64]
        np.save(out_fn,fea)
print(i+1,'/',len(wavlst),end='\r')

In [6]:
def ret_f0_dur(f0,lab):
    labtoks=lab[1]
    retval=[]
    for i in range(len(labtoks)):
        #print(labtoks[i])
        st=int(labtoks[i][0]*100)
        et=int(labtoks[i][1]*100)
        #print(st,et,f0[st:et])
        groups = groupby(f0[st:et], lambda x: x == 0.0)
        result = [list(group) for is_delimiter, group in groups if not is_delimiter]
        #print(len(result))
        durlens=[len(x) for x in result]
        #print(durlens)
        npdurlens=np.array(durlens)
        npdurlens.sort()
        #print(npdurlens[-1])
        if(len(npdurlens)>0):
            retval.append(npdurlens[-1])
        else:
            retval.append(0)
    return retval
aishell3f0path='/workspace/mnt/slave1/data/AIshell3_slmtk/syllable_f0/'
hakkaf0path='/workspace/mnt/slave3/HAT-corpus/Hakka_ASR_four_counties/f0/'
labs=[]
if(os.path.isfile('data/labels.pkl')):
    labs=pickle.load(open("data/labels.pkl",'rb'))
    print('load labels.pkl done.')
else:
    cmd_out=''
    no_label_list=[]
    
    TRlabs=pickle.load(open("../tone_rec_large/data/labels.pkl",'rb'))
    for i in range(len(TRlabs)):
        if(TRlabs[i][0][:3])=='SSB':
            print(' '*len(cmd_out),end='\r')
            cmd_out="parserlab {}/{} {}".format(i+1,len(TRlabs),TRlabs[i][0])
            print(cmd_out,end='\r')
            labs.append(TRlabs[i])
            with open(aishell3f0path+TRlabs[i][0][:-4]+'/'+TRlabs[i][0]+'.f0','r') as f:
                f0=[float(line.strip()) for line in f.readlines()]
            durlens = ret_f0_dur(f0,TRlabs[i])
            for j in range(len(labs[-1][1])):
                labs[-1][1][j][2]='M'+str(labs[-1][1][j][2])
                labs[-1][1][j].append(durlens[j])
        else:
            continue
    #Hakkalab
    for i in range(len(Hakkalab)):
        print(' '*len(cmd_out),end='\r')
        try:
            #print(Hakkalab[i])
            alignfilename=os.path.basename(Hakkalab[i]).split('.')[0]
            cmd_out="parserlab {}/{} {}".format(i+1,len(Hakkalab),alignfilename)
            print(cmd_out,end='\r')
            with open(json_dict[alignfilename],'r',encoding='utf8') as f:
                hakka_pinyin=json.load(f)['客語拼音']
            syllables,tones=get_syllable_tokens(hakka_pinyin)
            align_sylwt=get_alignment(Hakkalab[i])
            #print(alignfilename)
            #print(syllables,len(syllables))
            #print(tones,len(tones))
            #print(align_sylwt,len(align_sylwt))
            trn_lab=[]
            for j in range(len(align_sylwt)):
                st=round(float(align_sylwt[j][0]),2)
                et=round(float(align_sylwt[j][1]),2)
                word=align_sylwt[j][2]
                t='H'+tones[j]
                trn_lab.append([st,et,t,word])
            labs.append([alignfilename, trn_lab])
            
            with open(hakkaf0path+labs[-1][0].split('-')[0]+'/'+labs[-1][0]+'.f0','r') as f:
                f0=[float(line.strip()) for line in f.readlines()]
            durlens = ret_f0_dur(f0,labs[-1])
            for j in range(len(labs[-1][1])):
                labs[-1][1][j].append(durlens[j])
        except:
            no_label_list.append(fn)
        #break
    
    with open("data/labels.pkl", "wb") as file:
        pickle.dump(labs, file)
    print('output labels.pkl done.')
print(len(labs))

load labels.pkl done.
190402


In [7]:
def get_label_data(label):
    f=np.load(fea_dir+'/{}_fea63.npy'.format(label[0]))[2:]
    #print(label[0],np.shape(f),np.shape(f[-2:-1,:]))
    fw=81
    while len(f)<81:
        f=np.concatenate((f,f[-1:,:]),axis=0)
        #print(np.shape(f))
    half_fw=int((fw-1)/2)
    idx_arr = np.array([[x] for x in range(-half_fw,half_fw+1)])
    #print(idx_arr)
    label_fea=[]
    label_ans=[]
    regions=label[1]
    ff=int(round(float(regions[-1][1])*100))
    ff = ff if ff>=81 else 81
    for i in range(len(regions)):
        sf=int(round(float(regions[i][0])*100))
        ef=int(round(float(regions[i][1])*100))
        cf=int((sf+ef)*0.5)
        lb = cf - half_fw
        rb = cf + half_fw + 1
        #print(sf,cf,ef)
        if(cf >= half_fw and len(f) - cf > half_fw):
            sf = sf - cf + half_fw
            ef = ef - cf + half_fw
        elif(len(f) - cf <= half_fw):
            sf = sf - (len(f) - fw)
            ef = ef - (len(f) - fw)
            rb = len(f)
            lb = rb - fw
        else:
            lb = 0
            rb = fw
            
        #if(half_fw>cf):
        #    lb=0
        #    rb=fw
        #elif(ff-half_fw<=cf):
        #    sf=sf-(ff-fw)
        #    ef=ef-(ff-fw)
        #    rb=ff
        #    lb=int(ff-fw)
        #else:
        #    sf=sf-cf+half_fw
        #    ef=ef-cf+half_fw
        #    lb=int(cf-half_fw)
        #    rb=int(lb+fw)
        identify_arr=np.array([[1 if x >= sf and x <= ef else 0] for x in range(fw)])
        #print(sf,ef,len(identify_arr))
        #print(sf,cf,ef,lb,rb)
        #break
        slice_f=f[lb:rb,:]
        #print(np.shape(idx_arr),np.shape(slice_f),np.shape(identify_arr))
        out_fea=np.concatenate((idx_arr,np.concatenate((identify_arr,slice_f),axis=1)),axis=1)
        #print(np.shape(out_fea))
        #print(out_fea)
        #break
        if(regions[i][2] in aveoe4_target):
            label_fea.append(out_fea)
            label_ans.append(aveoe4_target[regions[i][2]]+[regions[i][4]])
    return label_fea,label_ans
#fea, ans = get_label_data(labs[0])
def get_batch_from_labels(labels, batch_size=128, shuffle=True):
    fea_batchs=[]
    ans_batchs=[]
    feas=[]
    anss=[]
    for i in range(len(labels)):
        #print(i,labels[i][0],end='   \r')
        label_fea,label_ans=get_label_data(labels[i])
        #print(len(label_ans),label_ans)
        feas.extend(label_fea)
        anss.extend(label_ans)
        #print(len(feas),len(anss))
        #break
    
    
    if(shuffle):
        npfeas=np.array(feas)
        npanss=np.array(anss)
        p = np.random.permutation(len(npanss))
        ret_feas=npfeas[p]
        ret_anss=npanss[p]
    else:
        ret_feas=np.array(feas)
        ret_anss=np.array(anss)
    
    ret_feas_lst=[]
    ret_anss_lst=[]
    x=0
    while x*batch_size<len(ret_feas):
        if((x+1)*batch_size>len(ret_feas)):
            ret_feas_lst.append(ret_feas[x*batch_size:])
            ret_anss_lst.append(ret_anss[x*batch_size:])
        else:
            ret_feas_lst.append(ret_feas[x*batch_size:(x+1)*batch_size])
            ret_anss_lst.append(ret_anss[x*batch_size:(x+1)*batch_size])
        x=x+1
    return ret_feas_lst, ret_anss_lst
#fea_batch, ans_batch = get_batch_from_labels(labs[:500])
#print(len(fea_batch),len(ans_batch),len(ans_batch[0]),len(ans_batch[-1]))

In [8]:
class data_loader:
    get_data_times=0
    def __init__(self, labels, chunk_size, batch_size=128, shuffle=True):
        self.labels=labels
        self.chunk_size=chunk_size
        self.num_a_round=math.ceil(len(labels)/chunk_size)
        self.shuffle=shuffle
        self.batch_size=batch_size
    def get_batch_data(self):
        if(self.get_data_times%self.num_a_round == 0 and self.shuffle):
            #print(self.get_data_times,'shuffle')
            random.shuffle(self.labels)
        
        if(self.get_data_times%self.num_a_round == self.num_a_round - 1):
            fea_batch, ans_batch = get_batch_from_labels(self.labels[(self.get_data_times%self.num_a_round)*self.chunk_size:], batch_size=self.batch_size, shuffle=self.shuffle)
        else:
            fea_batch, ans_batch = get_batch_from_labels(self.labels[(self.get_data_times%self.num_a_round)*self.chunk_size:((self.get_data_times%self.num_a_round)+1)*self.chunk_size], batch_size=self.batch_size, shuffle=self.shuffle)
        #print(self.get_data_times,self.get_data_times%self.num_a_round,len(fea_batch))
        self.get_data_times=self.get_data_times+1
        return fea_batch, ans_batch
#train_data_loader = data_loader(labs[:round(len(labs)*0.9)], 500)
#dev_data_loader = data_loader(labs[round(len(labs)*0.9):], 500)

In [9]:
def loss(model, x, y, training):
    # training=training is needed only if there are layers with different
    # behavior during training versus inference (e.g. Dropout).
    y_1,y_2 = model(x, training=training)
    return tf.keras.losses.MSE(y_true=y[:,:4], y_pred=y_1), tf.keras.losses.MSE(y_true=y[:,4:5], y_pred=y_2)
def grad(model, inputs, targets, dropout):
    with tf.GradientTape() as tape:
        PPC_loss, dur_loss = loss(model, inputs, targets, training=dropout)
        total_loss = tf.reduce_mean(PPC_loss) + tf.reduce_mean(dur_loss)
        return [PPC_loss, dur_loss], tape.gradient(total_loss, model.trainable_variables)
def my_fit(model, train_labels, epoch_num, batch_size, chunk_size, learning_rate, shuflag=True):
    train_data_loader = data_loader(train_labels, chunk_size, batch_size, shuflag)
    temp_output=""
    min_loss=10000000.0
    last_chunk_loss=10000000.0
    best_weight=model.get_weights()
    
    for epoch_idx in range(epoch_num):
        start_seconds = time.time()
        epoch_PPC_loss = tf.keras.metrics.Mean()
        epoch_dur_loss = tf.keras.metrics.Mean()
        for chunk_idx in range(train_data_loader.num_a_round):
            fea_batch, ans_batch=train_data_loader.get_batch_data()
            for mini_batch_idx in range(len(ans_batch)):
                loss_value, grads = grad(model, fea_batch[mini_batch_idx], ans_batch[mini_batch_idx], True)
                if(np.isnan(loss_value[0].numpy()).any() or np.isnan(loss_value[1].numpy()).any()):
                    model.load_weights('models/temp/MH_PPC2/PPC_model')
                    model.optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5)
                    continue
                model.optimizer.apply_gradients(zip(grads, model.trainable_variables))
                epoch_PPC_loss.update_state(loss_value[0])
                epoch_dur_loss.update_state(loss_value[1])
                print(" "*(len(temp_output)+1), end='\r')
                end_seconds = time.time()
                cost_time=round(end_seconds-start_seconds)
                temp_output="{}s, epoch {}, chunk {}/{}, mini_batch {}/{} epoch_PPC_loss={:.7f} last_chunk_loss={:.7f} epoch_dur_loss={:.7f}".format(  cost_time,
                                                                                                                                          epoch_idx+1, 
                                                                                                                                          chunk_idx+1, 
                                                                                                                                          train_data_loader.num_a_round, 
                                                                                                                                          mini_batch_idx+1, 
                                                                                                                                          len(ans_batch),
                                                                                                                                          epoch_PPC_loss.result(),
                                                                                                                                          last_chunk_loss,
                                                                                                                                          epoch_dur_loss.result())
                print(temp_output, end='                                            \r')
            
            if(chunk_idx > 0 and last_chunk_loss > epoch_PPC_loss.result()):
                os.system('mkdir -p models/temp/MH_PPC2')
                os.system('cp -r models/temp/MH_PPC/* models/temp/MH_PPC2/')
                last_chunk_loss = epoch_PPC_loss.result()
            model.save_weights('models/temp/MH_PPC/PPC_model')
            with open('MH_PPC.log','a') as f:
                f.write("\t{}\n".format(temp_output))
        #########################################################################################################################################################
        if(min_loss>epoch_PPC_loss.result()):
            min_loss=epoch_PPC_loss.result()
            model.save_weights('models/MH_PPC/PPC_model')
            temp_output=temp_output+"\tsaved."
            best_weight=model.get_weights()
        else:
            model.set_weights(best_weight)
        end_seconds = time.time()
        cost_time=round(end_seconds-start_seconds)
        temp_output="{}s,\t".format(cost_time)+temp_output
        print(temp_output)
        with open('MH_PPC.log','a') as f:
            f.write("{}\n".format(temp_output))

In [10]:
d_model=256
lindim=512
numhead=2
dt=0.15
x=tf.keras.Input(shape=[81, 65])
xb=tf.keras.layers.BatchNormalization()(x)
ex=tf.keras.layers.Dense(d_model, activation='elu')(xb)
qx=tf.keras.layers.Dense(d_model, activation='elu')(ex)
vx=tf.keras.layers.Dense(d_model, activation='elu')(ex)
kx=tf.keras.layers.Dense(d_model, activation='elu')(ex)
qxb=tf.keras.layers.BatchNormalization()(qx)
vxb=tf.keras.layers.BatchNormalization()(vx)
kxb=tf.keras.layers.BatchNormalization()(kx)
dqx=tf.keras.layers.Dropout(dt)(qxb)
dvx=tf.keras.layers.Dropout(dt)(vxb)
dkx=tf.keras.layers.Dropout(dt)(kxb)
mha = tf.keras.layers.MultiHeadAttention(num_heads=numhead, key_dim=d_model,dropout=dt)(dqx,dkx,dvx)
BAmha=tf.keras.layers.BatchNormalization()(mha+ex)
linear_t = tf.keras.layers.Dense(lindim, activation='elu')(BAmha)
linear = tf.keras.layers.Dense(d_model, activation='elu')(linear_t)
xb2=tf.keras.layers.BatchNormalization()(BAmha+linear)
dxb2=tf.keras.layers.Dropout(dt)(xb2)
qx2=tf.keras.layers.Dense(d_model, activation='elu')(dxb2)
vx2=tf.keras.layers.Dense(d_model, activation='elu')(dxb2)
kx2=tf.keras.layers.Dense(d_model, activation='elu')(dxb2)
qx2b=tf.keras.layers.BatchNormalization()(qx2)
vx2b=tf.keras.layers.BatchNormalization()(vx2)
kx2b=tf.keras.layers.BatchNormalization()(kx2)
dqx2=tf.keras.layers.Dropout(dt)(qx2b)
dvx2=tf.keras.layers.Dropout(dt)(vx2b)
dkx2=tf.keras.layers.Dropout(dt)(kx2b)
mha2 = tf.keras.layers.MultiHeadAttention(num_heads=numhead, key_dim=d_model,dropout=dt)(dqx2,dkx2,dvx2)
BAmha2=tf.keras.layers.BatchNormalization()(mha2+xb2)
linear2_t = tf.keras.layers.Dense(lindim, activation='elu')(BAmha2)
linear2 = tf.keras.layers.Dense(d_model, activation='elu')(linear2_t)
xb3=tf.keras.layers.BatchNormalization()(BAmha2+linear2)
dxb3=tf.keras.layers.Dropout(dt)(xb3)
qx3=tf.keras.layers.Dense(d_model, activation='elu')(dxb3)
vx3=tf.keras.layers.Dense(d_model, activation='elu')(dxb3)
kx3=tf.keras.layers.Dense(d_model, activation='elu')(dxb3)
qx3b=tf.keras.layers.BatchNormalization()(qx3)
vx3b=tf.keras.layers.BatchNormalization()(vx3)
kx3b=tf.keras.layers.BatchNormalization()(kx3)
dqx3=tf.keras.layers.Dropout(dt)(qx3b)
dvx3=tf.keras.layers.Dropout(dt)(vx3b)
dkx3=tf.keras.layers.Dropout(dt)(kx3b)
mha3 = tf.keras.layers.MultiHeadAttention(num_heads=numhead, key_dim=d_model,dropout=dt)(dqx3,dkx3,dvx3)
BAmha3=tf.keras.layers.BatchNormalization()(mha3+xb3)
linear3 = tf.keras.layers.Dense(lindim, activation='elu')(BAmha3[:,0,:])
#xb4=tf.keras.layers.BatchNormalization()(linear3)
#dxb4=tf.keras.layers.Dropout(0.025)(xb4)
#mid_mha=tf.keras.layers.Flatten()(linear3)
out1 = tf.keras.layers.Dense(4)(linear3)

linear_flatten=tf.keras.layers.Flatten()(linear)
flattenL=tf.keras.layers.Dense(d_model, activation='elu')(linear_flatten)
out2 = tf.keras.layers.Dense(1)(flattenL)

model = tf.keras.Model(inputs = x, outputs = [out1,out2])

2026-03-28 16:30:37.324909: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1883] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9250 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:17:00.0, compute capability: 7.5


In [11]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 81, 65)]             0         []                            
                                                                                                  
 batch_normalization (Batch  (None, 81, 65)               260       ['input_1[0][0]']             
 Normalization)                                                                                   
                                                                                                  
 dense (Dense)               (None, 81, 256)              16896     ['batch_normalization[0][0]'] 
                                                                                                  
 dense_1 (Dense)             (None, 81, 256)              65792     ['dense[0][0]']           

In [12]:
model.load_weights('models/MH_PPC/PPC_model')
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5), loss='mse', metrics=['accuracy'])

In [ ]:
my_fit(model=model,train_labels=labs, epoch_num=100,batch_size=384, chunk_size=5000, learning_rate=5e-5)

2026-03-28 16:30:48.317177: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x5fb1bcff08f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-28 16:30:48.317219: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2026-03-28 16:30:48.320965: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-28 16:30:48.334015: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8907
2026-03-28 16:30:48.405992: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1849s,	1849s, epoch 1, chunk 39/39, mini_batch 16/16 epoch_PPC_loss=0.0008245 last_chunk_loss=0.0008127 epoch_dur_loss=0.4494067	saved.                                     
1749s,	1748s, epoch 2, chunk 39/39, mini_batch 15/15 epoch_PPC_loss=0.0008216 last_chunk_loss=0.0007942 epoch_dur_loss=0.4447587	saved.                                
1751s,	1750s, epoch 3, chunk 39/39, mini_batch 16/16 epoch_PPC_loss=0.0008178 last_chunk_loss=0.0007942 epoch_dur_loss=0.4315629	saved.                                
1752s,	1751s, epoch 4, chunk 39/39, mini_batch 16/16 epoch_PPC_loss=0.0008159 last_chunk_loss=0.0007942 epoch_dur_loss=0.4315380	saved.                                


In [13]:
fea, ans = get_label_data(labs[0])

In [15]:
res=model.predict(np.array(fea))

1/1 [==============================] - 1s 658ms/step


In [26]:
print(np.array(ans),'\n\n',res)

[[ 5.24617300e+00  4.04906884e-03  2.45491770e-02 -9.27637412e-03]
 [ 5.24617300e+00  4.04906884e-03  2.45491770e-02 -9.27637412e-03]
 [ 5.35935741e+00 -8.57866300e-02 -2.80325872e-02  1.97749802e-02]
 [ 5.24617300e+00  4.04906884e-03  2.45491770e-02 -9.27637412e-03]
 [ 5.35935741e+00 -8.57866300e-02 -2.80325872e-02  1.97749802e-02]
 [ 5.46909305e+00  1.10121000e-02 -1.45683239e-02 -5.78306388e-03]
 [ 5.46909305e+00  1.10121000e-02 -1.45683239e-02 -5.78306388e-03]
 [ 5.46909305e+00  1.10121000e-02 -1.45683239e-02 -5.78306388e-03]
 [ 5.46909305e+00  1.10121000e-02 -1.45683239e-02 -5.78306388e-03]
 [ 5.46909305e+00  1.10121000e-02 -1.45683239e-02 -5.78306388e-03]
 [ 5.46909305e+00  1.10121000e-02 -1.45683239e-02 -5.78306388e-03]
 [ 5.46909305e+00  1.10121000e-02 -1.45683239e-02 -5.78306388e-03]] 

 [[ 5.39716005e+00 -4.64088097e-02 -1.57616120e-02  1.18465777e-02]
 [ 5.31733704e+00 -7.35636428e-02  1.47481682e-04  1.27945896e-02]
 [ 5.39540529e+00 -2.41907593e-02 -1.70854735e-03 -2.30124

In [17]:
np.array(ans)[:,:4]

array([[ 5.49341593e+00,  1.80786657e-02, -7.09353493e-03,
        -5.98973049e-03],
       [ 5.28116466e+00, -1.28633455e-01,  2.12441495e-03,
         1.90413281e-02],
       [ 5.30775951e+00, -9.67147671e-02,  3.52695882e-03,
         1.21849745e-02],
       [ 5.15217286e+00, -8.48671819e-03,  4.91156855e-02,
        -1.19366733e-02],
       [ 5.49341593e+00,  1.80786657e-02, -7.09353493e-03,
        -5.98973049e-03],
       [ 5.51971473e+00,  1.18724080e-02, -4.54205008e-03,
        -3.05479306e-03],
       [ 5.10452305e+00, -1.30883285e-01,  2.50564005e-02,
         1.76348863e-02],
       [ 5.49341593e+00,  1.80786657e-02, -7.09353493e-03,
        -5.98973049e-03],
       [ 5.49341593e+00,  1.80786657e-02, -7.09353493e-03,
        -5.98973049e-03],
       [ 5.10452305e+00, -1.30883285e-01,  2.50564005e-02,
         1.76348863e-02],
       [ 5.49341593e+00,  1.80786657e-02, -7.09353493e-03,
        -5.98973049e-03],
       [ 5.10452305e+00, -1.30883285e-01,  2.50564005e-02,
      

In [13]:
gc.collect()

912